# Train Tabular SARSA

SARSA is an on-policy tabular algorithm that updates

$$Q(s_t,a_t)\leftarrow Q(s_t,a_t)+\alpha\left[r_{t+1}+\gamma(1-d_t)Q(s_{t+1},a_{t+1})-Q(s_t,a_t)\right].$$

Here $\alpha$ is the learning rate, $\gamma$ the discount factor, $d_t$ the terminal indicator, and $a_{t+1}$ the next behavior action. This notebook trains it on `Taxi-v4` with epsilon-greedy exploration.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import SARSA, SARSAConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "Taxi-v4"

SARSA learns on-policy: the epsilon-greedy action in each update is the action actually taken on the following step.

In [ ]:
env = gym.make(ENV_ID)
config = SARSAConfig(
    learning_rate=0.2,
    gamma=0.95,
    exploration_steps=80_000,
)

agent = SARSA(env, config=config)
agent.learn(total_timesteps=100_000)
env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(100, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"SARSA training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions. Each Taxi episode starts with a new taxi, passenger, and destination configuration.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human")
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")